In [1]:
import os
import numpy as np
import torch

from BudaOCR.Config import CHARSET
from BudaOCR.Networks import EasterNetwork
from BudaOCR.Encoder import StackEncoder, WylieEncoder
from BudaOCR.Trainer import OCRTrainer

from BudaOCR.Utils import (
    accumulate_distributions,
    build_distribution_from_file,
    build_data_paths,
    create_dir,
    read_stack_file,
    shuffle_data
    )


print(torch.__version__)
torch.cuda.empty_cache()

print(torch.cuda.is_available())

/Users/eric/Desktop/Projects/Python/tibetan-ocr-training/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.10.0
False


In [2]:
image_width = 3200
image_height = 100
wylie_encoder = WylieEncoder(CHARSET)

stack_file = "tib-stacks_v2.txt"
stacks = read_stack_file(stack_file)
stack_encoder = StackEncoder(stacks)

# setting the encoder to be used
encoder = wylie_encoder

# train params
batch_size = 32
workers = 4
network = EasterNetwork("Easter2b", image_width, image_height, num_classes=encoder.num_classes)

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


building ctcvocab: 84
building ctcvocab: 10376
Using MPS compute backend
Network -> Architecture: Easter2b, input width: 3200, input height: 100


#### Single Dataset Training

In [3]:
# local dir
dataset_path = "/Users/eric/Desktop/Data/Namgyal_LineDataset"
image_paths, label_paths = build_data_paths(dataset_path, img_file_ext="jpg")
image_paths, label_paths = shuffle_data(image_paths, label_paths)

print(f"Images: {len(image_paths)}, Labels: {len(label_paths)}")

output_dir = os.path.join("Output")
create_dir(output_dir)

ValueError: not enough values to unpack (expected 2, got 0)

In [ ]:
ocr_trainer = OCRTrainer(
    network=network,
    label_encoder=encoder,
    workers=workers, 
    image_width=image_width,
    image_height=image_height,
    batch_size=batch_size, 
    output_dir=output_dir, 
    preload_labels=True
    )

ocr_trainer.init(image_paths, label_paths)

In [ ]:
num_epochs = 1
ocr_trainer.train(epochs=num_epochs, check_cer=True, export_onnx=True, silent=False)

#### Evaluate on Test set

In [ ]:
# run evaluation
cer_scores = ocr_trainer.evaluate()
cer_values = list(cer_scores.values())

score_file = os.path.join(ocr_trainer.output_dir, "cer_scores.txt")

with open(score_file, "w", encoding="utf-8") as f:
    for sample, value in cer_scores.items():
        f.write(f"{sample} - {value}\n")

cer_summary_file = os.path.join(ocr_trainer.output_dir, "cer_summary.txt")

mean_cer = np.mean(cer_values)
max_cer = np.max(cer_values)
min_cer = np.min(cer_values)

with open(cer_summary_file, "w", encoding="utf-8") as f:
    f.write(f"Mean CER: {mean_cer}\n")
    f.write(f"Max CER: {max_cer}\n")
    f.write(f"Min CER: {min_cer}")


print(f"Mean CER: {mean_cer}")
print(f"Max CER: {max_cer}")
print(f"Min CER: {min_cer}")

#### Train from Distribution file

In [ ]:
dataset_path = "../Data/Karmapa8"
distr_file = f"{dataset_path}/data.distribution"
distribution = build_distribution_from_file(distr_file, dataset_path)

In [ ]:
output_dir = os.path.join(dataset_path, "Output")
create_dir(output_dir)


ocr_trainer = OCRTrainer(
    network=network,
    label_encoder=encoder,
    workers=workers, 
    image_width=image_width,
    image_height=image_height,
    batch_size=16, 
    output_dir=output_dir, 
    preload_labels=True
    )

assert (distribution is not None)
ocr_trainer.init_from_distribution(distribution)

In [ ]:
num_epochs = 12
ocr_trainer.train(epochs=num_epochs, check_cer=True, export_onnx=True, silent=False)

#### Training multiple distributions

In [ ]:
data_root = "../home"
distributions = ["DergeTenjur", "LhasaKanjur", "Karmapa8", "LithangKanjur"]
distribution = accumulate_distributions(data_root, distributions)

In [ ]:
output_dir = "Output"
create_dir(output_dir)


ocr_trainer = OCRTrainer(
    network=network,
    label_encoder=encoder,
    workers=workers, 
    image_width=image_width,
    image_height=image_height,
    batch_size=32, 
    output_dir=output_dir, 
    preload_labels=True
    )

assert (distribution is not None)
ocr_trainer.init_from_distribution(distribution)

In [ ]:
num_epochs = 12
ocr_trainer.train(epochs=num_epochs, check_cer=True, export_onnx=True, silent=False)